# Session 5 — Options Pricing: Black-Scholes, Monte Carlo & Real Options
## Risk Modelling Course

---

### Learning Objectives
By the end of this session you will be able to:
1. Implement the **Black-Scholes formula** analytically in Python and understand each component
2. Compute and visualise the **Greeks** (Delta, Gamma, Vega, Theta, Rho)
3. Price European options by **Monte Carlo simulation** (GBM path simulation)
4. Price **path-dependent options** (Asian options) that have no closed-form solution
5. Demonstrate that MC prices **converge** to the BS price as simulation paths increase
6. Apply **implied volatility** concepts and understand the volatility smile
7. Introduce **Real Options** — frame corporate investments as financial options

---

### Core idea — Geometric Brownian Motion (GBM)
Black-Scholes assumes the underlying asset follows:

$$dS = \mu S \, dt + \sigma S \, dW_t$$

Where $dW_t = \epsilon \sqrt{dt}$ is a Wiener process increment ($\epsilon \sim N(0,1)$).

In discrete time (for simulation) this becomes:

$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t}\, \epsilon\right]$$

The **risk-neutral** version replaces $\mu$ with $r$ (the risk-free rate) — a foundational result.


## 0. Imports & Configuration

In [ ]:

import warnings
warnings.filterwarnings("ignore")

# ── Numerics ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq   # for implied vol solver

# ── Market data ───────────────────────────────────────────────────────────────
import yfinance as yf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)

print("All libraries loaded.")


## 1. The Black-Scholes Formula (Analytical)

The Black-Scholes formula prices a **European** option (can only be exercised at expiry):

$$C = S_0 \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$P = K e^{-rT} \cdot N(-d_2) - S_0 \cdot N(-d_1)$$

Where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

- $S_0$ = current stock price
- $K$ = strike price
- $T$ = time to maturity (in years)
- $r$ = continuously compounded risk-free rate
- $\sigma$ = annual volatility of the underlying
- $N(\cdot)$ = standard normal CDF


In [ ]:

def black_scholes(S: float, K: float, T: float, r: float,
                  sigma: float, option: str = "call") -> dict:
    """
    Compute Black-Scholes price and Greeks for a European option.

    Parameters
    ----------
    S      : Current stock price
    K      : Strike price
    T      : Time to maturity in years
    r      : Continuously compounded risk-free rate (annual)
    sigma  : Annual volatility (e.g. 0.20 for 20%)
    option : 'call' or 'put'

    Returns
    -------
    dict with keys: price, d1, d2, delta, gamma, vega, theta, rho
    """
    # ── Protect against T=0 ───────────────────────────────────────────────────
    if T <= 0:
        intrinsic = max(S - K, 0) if option == "call" else max(K - S, 0)
        return {"price": intrinsic}

    # ── d1 and d2 ─────────────────────────────────────────────────────────────
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # ── Price ──────────────────────────────────────────────────────────────────
    if option == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    # ── Greeks ────────────────────────────────────────────────────────────────
    # Delta: sensitivity of price to ±$1 move in S
    delta = norm.cdf(d1) if option == "call" else norm.cdf(d1) - 1

    # Gamma: rate of change of Delta w.r.t. S (same for calls and puts)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    # Vega: sensitivity to ±1% change in sigma (reported per 1% move)
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100

    # Theta: time decay per calendar day (reported as daily loss)
    if option == "call":
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        ) / 365
    else:
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        ) / 365

    # Rho: sensitivity to ±1% change in risk-free rate
    if option == "call":
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)  / 100
    else:
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    return {
        "price": price, "d1": d1, "d2": d2,
        "delta": delta, "gamma": gamma,
        "vega":  vega,  "theta": theta, "rho": rho
    }


# ── Example pricing ───────────────────────────────────────────────────────────
# AAPL-like parameters: stock at 180, ATM call, 3 months to expiry, 20% vol
params = dict(S=180, K=180, T=0.25, r=0.05, sigma=0.20)

call_result = black_scholes(**params, option="call")
put_result  = black_scholes(**params, option="put")

print(f"{'':=<55}")
print(f"  Black-Scholes Pricing Example")
print(f"  S={params['S']}, K={params['K']}, T={params['T']}yr, r={params['r']:.0%}, σ={params['sigma']:.0%}")
print(f"{'':=<55}")
for key in ["price", "delta", "gamma", "vega", "theta", "rho"]:
    print(f"  CALL {key:>6}: {call_result[key]:>9.4f}    PUT {key:>6}: {put_result[key]:>9.4f}")
print(f"{'':=<55}")
print()
print("Put-Call Parity check (C - P should equal S - K·e^{-rT}):")
lhs = call_result["price"] - put_result["price"]
rhs = params["S"] - params["K"] * np.exp(-params["r"] * params["T"])
print(f"  C - P = {lhs:.4f}   |   S - Ke^(-rT) = {rhs:.4f}   ✓" if abs(lhs-rhs) < 1e-8
      else f"  MISMATCH: {lhs:.4f} vs {rhs:.4f}")


## 2. Visualising the Greeks

Greeks tell traders *how* the option price changes with each input.
Understanding them is essential for **hedging** (Delta hedging) and **risk management**.

| Greek | What it measures | Intuition |
|-------|-----------------|-----------|
| Delta | ∂Price/∂S | Like a hedge ratio — a delta of 0.5 means price moves 50p per £1 in stock |
| Gamma | ∂²Price/∂S² | Rate of change of delta; largest near ATM options |
| Vega  | ∂Price/∂σ | How much the option gains/loses per 1% vol rise |
| Theta | ∂Price/∂t | Time decay — options lose value as expiry approaches |


In [ ]:

# ── Parameter grid: vary spot price from 50% to 150% of strike ────────────────
K, T, r, sigma = 100, 0.5, 0.05, 0.25
spot_range = np.linspace(50, 150, 300)

# Build a DataFrame for call and put Greeks across spot prices using chaining
def greeks_df(option_type):
    """Compute all Greeks across a spot price range."""
    records = [black_scholes(S=s, K=K, T=T, r=r, sigma=sigma, option=option_type)
               for s in spot_range]
    return (
        pd.DataFrame(records)
        .assign(Spot=spot_range, Option=option_type)
    )

greeks_all = pd.concat([greeks_df("call"), greeks_df("put")], ignore_index=True)

# ── 4-panel plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
greek_specs = [
    ("price", "Option Price (£)",         "Price vs Spot"),
    ("delta", "Delta",                     "Delta vs Spot"),
    ("gamma", "Gamma",                     "Gamma vs Spot (identical for C & P)"),
    ("vega",  "Vega (£ per 1% σ move)",   "Vega vs Spot"),
]

for ax, (greek, ylabel, title) in zip(axes.flatten(), greek_specs):
    for opt_type, grp in greeks_all.groupby("Option"):
        ax.plot(grp["Spot"], grp[greek], lw=2,
                label=opt_type.capitalize(),
                ls="-" if opt_type == "call" else "--")

    ax.axvline(K, color="grey", lw=1, ls=":", label=f"Strike K={K}")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Spot Price (S)")
    ax.set_ylabel(ylabel)
    ax.legend()

fig.suptitle(f"Black-Scholes Greeks  |  K={K}, T={T}yr, r={r:.0%}, σ={sigma:.0%}",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 3. Simulating Stock Price Paths (GBM)

Before pricing options by simulation, let's visualise what GBM looks like.

The key discretisation (Euler-Maruyama scheme):
$$S_{t+1} = S_t \exp\!\left[(r - \tfrac{1}{2}\sigma^2)\,\Delta t + \sigma\sqrt{\Delta t}\,\epsilon_t\right]$$

We use **vectorised NumPy** rather than Python loops — this is critical for speed when
running 100,000 paths.


In [ ]:

def simulate_gbm(S0: float, r: float, sigma: float,
                 T: float, n_steps: int, n_paths: int) -> np.ndarray:
    """
    Simulate GBM paths using vectorised NumPy operations.

    Returns
    -------
    np.ndarray of shape (n_steps + 1, n_paths)
    Each column is one simulated price path.
    """
    dt = T / n_steps

    # ── Draw all random increments at once (much faster than looping) ─────────
    # Shape: (n_steps, n_paths)
    Z = np.random.standard_normal((n_steps, n_paths))

    # ── Compute log-return increments ─────────────────────────────────────────
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    # ── Cumulative sum and exponentiation to recover price levels ──────────────
    # np.vstack prepends a row of zeros for S0
    log_paths = np.vstack([np.zeros(n_paths), np.cumsum(increments, axis=0)])
    paths = S0 * np.exp(log_paths)   # shape: (n_steps + 1, n_paths)

    return paths


# ── Illustrative simulation: 200 paths, 1 year ────────────────────────────────
S0, r, sigma, T = 100, 0.05, 0.25, 1.0
N_STEPS_PLOT = 252   # daily steps
paths_plot = simulate_gbm(S0, r, sigma, T, N_STEPS_PLOT, n_paths=200)

time_axis = np.linspace(0, T, N_STEPS_PLOT + 1)

fig, ax = plt.subplots(figsize=(13, 5))

# Plot individual paths with low alpha
ax.plot(time_axis, paths_plot[:, :50], lw=0.6, alpha=0.3, color="steelblue")

# Highlight 5th and 95th percentile bands
p5  = np.percentile(paths_plot, 5,  axis=1)
p95 = np.percentile(paths_plot, 95, axis=1)
ax.fill_between(time_axis, p5, p95, alpha=0.2, color="steelblue", label="5th–95th percentile")
ax.plot(time_axis, np.median(paths_plot, axis=1), color="navy", lw=2, label="Median path")
ax.axhline(S0, color="grey", lw=0.8, ls="--", label=f"S₀ = {S0}")

ax.set_title(f"GBM Simulated Stock Paths  |  S₀={S0}, r={r:.0%}, σ={sigma:.0%}, T={T}yr",
             fontweight="bold")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Stock Price (£)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFinal price distribution (n=200 paths):")
final_prices = pd.Series(paths_plot[-1, :], name="Final Price")
print(final_prices.describe().to_string())


## 4. Monte Carlo Option Pricing — European Options

**Algorithm:**
1. Simulate $N$ terminal stock prices $S_T$ under the risk-neutral measure
2. Compute the payoff for each path: $\max(S_T - K, 0)$ for a call
3. Average the payoffs and discount back at the risk-free rate

$$C_{MC} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K,\, 0)$$

This works for **any** payoff function — the real power of simulation.


In [ ]:

def mc_european(S: float, K: float, T: float, r: float,
                sigma: float, n_sims: int = 100_000,
                option: str = "call") -> dict:
    """
    Price a European option by Monte Carlo simulation (one-step, terminal price only).

    Returns dict with: price, std_error, ci_lower, ci_upper
    """
    # ── Simulate terminal stock prices (one-step trick — no need for full paths) ─
    Z  = np.random.standard_normal(n_sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    # ── Compute payoffs ───────────────────────────────────────────────────────
    if option == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)

    # ── Discount and compute 95% confidence interval ──────────────────────────
    discount_factor = np.exp(-r * T)
    price      = discount_factor * payoffs.mean()
    std_error  = discount_factor * payoffs.std() / np.sqrt(n_sims)
    ci_lower   = price - 1.96 * std_error
    ci_upper   = price + 1.96 * std_error

    return {"price": price, "std_error": std_error,
            "ci_lower": ci_lower, "ci_upper": ci_upper}


# ── Compare MC vs BS for a range of strikes ───────────────────────────────────
S0, T, r, sigma = 100, 0.5, 0.05, 0.20
strikes = np.arange(70, 135, 5)

rows = []
for K in strikes:
    bs  = black_scholes(S=S0, K=K, T=T, r=r, sigma=sigma, option="call")
    mc  = mc_european(S=S0, K=K, T=T, r=r, sigma=sigma, n_sims=100_000, option="call")
    rows.append({
        "Strike": K,
        "BS Price":  bs["price"],
        "MC Price":  mc["price"],
        "MC Std Err": mc["std_error"],
        "Difference": mc["price"] - bs["price"],
        "Moneyness": "ITM" if S0 > K else ("ATM" if S0 == K else "OTM"),
    })

comparison = pd.DataFrame(rows).set_index("Strike")
print("Black-Scholes vs Monte Carlo (100,000 paths):")
comparison[["BS Price", "MC Price", "MC Std Err", "Difference", "Moneyness"]]


In [ ]:

# ── Visualise the convergence of MC price to BS price ─────────────────────────
# As we increase the number of simulation paths, MC converges to the analytical value

K_atm = 100   # At-the-money call
bs_price = black_scholes(S=S0, K=K_atm, T=T, r=r, sigma=sigma, option="call")["price"]

path_counts = [100, 500, 1_000, 2_000, 5_000, 10_000, 25_000, 50_000, 100_000]
n_repeats   = 30   # run 30 replicates at each path count to see variance

# ── Build results using list comprehension + method chaining ──────────────────
conv_results = (
    pd.DataFrame([
        {
            "n_paths": n,
            "replicate": rep,
            "mc_price": mc_european(S=S0, K=K_atm, T=T, r=r, sigma=sigma,
                                    n_sims=n, option="call")["price"]
        }
        for n in path_counts
        for rep in range(n_repeats)
    ])
    .assign(error=lambda df: df["mc_price"] - bs_price)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: price convergence ────────────────────────────────────────────────────
sns.lineplot(data=conv_results, x="n_paths", y="mc_price",
             estimator="mean", errorbar="sd",
             ax=axes[0], color="steelblue", lw=2)
axes[0].axhline(bs_price, color="red", lw=2, ls="--", label=f"BS price = {bs_price:.3f}")
axes[0].set_xscale("log")
axes[0].set_title("MC Price Convergence to BS Analytical Price", fontweight="bold")
axes[0].set_xlabel("Number of Simulated Paths (log scale)")
axes[0].set_ylabel("Option Price (£)")
axes[0].legend()

# ── Right: absolute error ──────────────────────────────────────────────────────
sns.lineplot(data=conv_results.assign(abs_error=lambda df: df["error"].abs()),
             x="n_paths", y="abs_error",
             estimator="mean", errorbar="sd",
             ax=axes[1], color="darkorange", lw=2)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Absolute Error vs Number of Paths", fontweight="bold")
axes[1].set_xlabel("Number of Simulated Paths (log scale)")
axes[1].set_ylabel("|MC Price − BS Price| (log scale)")

plt.suptitle("Monte Carlo Convergence Analysis  |  ATM Call, S=K=100, T=0.5yr, σ=20%",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: error shrinks as 1/√N — doubling accuracy requires 4× more paths.")


## 5. Path-Dependent Options — Asian Options

The real power of Monte Carlo is pricing options with **no closed-form solution**.

An **Asian call option** pays based on the *average* stock price over the life of the option:

$$\text{Payoff} = \max\left(\bar{S} - K,\, 0\right), \quad \bar{S} = \frac{1}{n}\sum_{t=1}^{n} S_t$$

This is widely used in commodity markets (e.g. oil derivatives) because a company buying
oil every month cares about the average price, not the price on one specific day.

There is no Black-Scholes equivalent formula — simulation is the primary pricing tool.


In [ ]:

def mc_asian_call(S: float, K: float, T: float, r: float,
                  sigma: float, n_steps: int = 252,
                  n_sims: int = 100_000) -> dict:
    """
    Price an arithmetic-average Asian call option by Monte Carlo.

    Parameters
    ----------
    n_steps : number of averaging points (e.g. 252 for daily averaging)

    Returns
    -------
    dict with price, std_error, ci_lower, ci_upper
    """
    # ── Simulate full price paths (shape: n_steps × n_sims) ──────────────────
    paths = simulate_gbm(S0=S, r=r, sigma=sigma, T=T,
                         n_steps=n_steps, n_paths=n_sims)

    # ── Average stock price for each path (column-wise mean, exclude t=0) ─────
    avg_price = paths[1:, :].mean(axis=0)   # arithmetic average of daily prices

    # ── Compute payoffs ───────────────────────────────────────────────────────
    payoffs = np.maximum(avg_price - K, 0)

    price     = np.exp(-r * T) * payoffs.mean()
    std_error = np.exp(-r * T) * payoffs.std() / np.sqrt(n_sims)

    return {
        "price":     price,
        "std_error": std_error,
        "ci_lower":  price - 1.96 * std_error,
        "ci_upper":  price + 1.96 * std_error,
    }


# ── Compare European vs Asian call prices across strikes ──────────────────────
S0, T, r, sigma = 100, 1.0, 0.05, 0.20

comparison_asian = (
    pd.DataFrame({"Strike": np.arange(80, 125, 5)})
    .assign(
        European_BS  = lambda df: df["Strike"].map(
            lambda K: black_scholes(S=S0, K=K, T=T, r=r, sigma=sigma, option="call")["price"]),
        Asian_MC     = lambda df: df["Strike"].map(
            lambda K: mc_asian_call(S=S0, K=K, T=T, r=r, sigma=sigma,
                                    n_steps=252, n_sims=50_000)["price"]),
    )
    .assign(Asian_Discount = lambda df: 1 - df["Asian_MC"] / df["European_BS"])
    .set_index("Strike")
)

print("European (BS) vs Asian (MC) Call Prices:")
print(comparison_asian.to_string())

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(comparison_asian.index, comparison_asian["European_BS"],
        "o-", lw=2, label="European Call (Black-Scholes)")
ax.plot(comparison_asian.index, comparison_asian["Asian_MC"],
        "s--", lw=2, label="Asian Call (Monte Carlo)")
ax.fill_between(comparison_asian.index,
                comparison_asian["Asian_MC"],
                comparison_asian["European_BS"],
                alpha=0.15, color="green", label="Asian discount")

ax.axvline(S0, color="grey", lw=1, ls=":", label="ATM (S=K=100)")
ax.set_title("European vs Asian Call Prices  |  S=100, σ=20%, T=1yr",
             fontweight="bold")
ax.set_xlabel("Strike Price (K)")
ax.set_ylabel("Option Price (£)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nKey insight: Asian calls are CHEAPER than European calls.")
print("Averaging reduces volatility of the payoff — so you pay less for the option.")


## 6. Implied Volatility & The Volatility Smile

**Implied volatility (IV)** is the value of $\sigma$ that makes the Black-Scholes formula
reproduce a market-observed option price. It is found by **inverting** the BS formula numerically.

$$C_{\text{market}} = \text{BS}(S, K, T, r, \sigma_{\text{implied}})$$

If Black-Scholes were perfectly correct, IV would be constant across all strikes.
In reality, it forms a **volatility smile** or **skew** — evidence that BS assumptions
(constant vol, no jumps) are violated.


In [ ]:

def implied_vol(market_price: float, S: float, K: float, T: float,
                r: float, option: str = "call",
                tol: float = 1e-6) -> float:
    """
    Compute implied volatility by numerically inverting the BS formula.
    Uses Brent's method (scipy.optimize.brentq) — robust and fast.

    Returns float (implied vol) or np.nan if no solution found.
    """
    # ── Check if price is within no-arbitrage bounds ───────────────────────────
    intrinsic = max(S - K * np.exp(-r * T), 0) if option == "call" else max(K * np.exp(-r * T) - S, 0)
    if market_price < intrinsic:
        return np.nan

    # ── Objective function: BS price - market price = 0 ──────────────────────
    objective = lambda sigma: black_scholes(S, K, T, r, sigma, option)["price"] - market_price

    try:
        return brentq(objective, 1e-6, 10.0, xtol=tol)  # search vol in [0.0001, 1000%]
    except ValueError:
        return np.nan


# ── Simulate a volatility smile by constructing market prices with a skew ─────
# We 'generate' synthetic market prices using a skewed vol surface as ground truth,
# then recover the implied vol for each strike. This mimics real market behaviour.

S0, T, r = 100, 0.5, 0.05
strikes = np.arange(70, 135, 2.5)

def true_vol_smile(K, S0, atm_vol=0.20, skew=0.002):
    """ Simplified parametric smile: vol increases as we go OTM. """
    moneyness = np.log(K / S0)
    return atm_vol + skew * moneyness**2 - 0.001 * moneyness  # quadratic + skew

rows_smile = []
for K in strikes:
    true_sigma = true_vol_smile(K, S0)
    mkt_price  = black_scholes(S0, K, T, r, true_sigma, option="call")["price"]
    imp_vol    = implied_vol(mkt_price, S0, K, T, r, option="call")
    rows_smile.append({"Strike": K, "True Vol": true_sigma, "Implied Vol": imp_vol,
                       "Market Price": mkt_price})

smile_df = pd.DataFrame(rows_smile).set_index("Strike")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Volatility smile ─────────────────────────────────────────────────────
axes[0].plot(smile_df.index, smile_df["Implied Vol"] * 100, "o-", lw=2,
             color="steelblue", label="Implied Vol (recovered)")
axes[0].axhline(20, color="grey", lw=1, ls="--", label="Flat vol = 20%")
axes[0].axvline(S0, color="orange", lw=1, ls=":", label="ATM")
axes[0].set_title("The Volatility Smile", fontweight="bold")
axes[0].set_xlabel("Strike Price")
axes[0].set_ylabel("Implied Volatility (%)")
axes[0].legend()

# ── Right: Option prices showing skew ─────────────────────────────────────────
bs_flat_prices = [black_scholes(S0, K, T, r, 0.20, "call")["price"] for K in strikes]
axes[1].plot(strikes, smile_df["Market Price"], "o-", lw=2, label="Market prices (skewed vol)")
axes[1].plot(strikes, bs_flat_prices, "s--", lw=2, label="BS prices (flat 20% vol)")
axes[1].set_title("Skewed Market Prices vs Flat-Vol BS", fontweight="bold")
axes[1].set_xlabel("Strike Price")
axes[1].set_ylabel("Call Option Price (£)")
axes[1].legend()

plt.suptitle("Implied Volatility and the Volatility Smile", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Real Options — Valuing Managerial Flexibility

### The concept
Financial options give the **right, but not the obligation**, to buy/sell an asset.
**Real options** extend this logic to corporate investment decisions:

| Real Option | Financial Analogy | Example |
|-------------|------------------|---------|
| Option to **expand** | Call option | Invest more if project succeeds |
| Option to **abandon** | Put option | Sell/exit if market deteriorates |
| Option to **defer**   | Call option | Wait for more information before committing |
| Option to **switch**  | Exchange option | Switch fuel source if gas price rises |

### Limitations of applying BS to real options
Black-Scholes assumes:
- Continuous trading in the underlying asset ❌ (can't trade a factory)
- Constant volatility ❌ (project cash flow vol changes over time)
- No dividends / intermediate cash flows ❌ (projects produce cash flows)

Despite these limitations, BS gives useful **order-of-magnitude** estimates and the
intuition is invaluable for communicating option value to management.


In [ ]:

def real_option_expand(PV_asset: float, I_expand: float,
                       T_years: float, r_rf: float,
                       sigma_project: float) -> dict:
    """
    Value an 'option to expand' as a European call option using Black-Scholes.

    Parameters
    ----------
    PV_asset       : Present value of the project's future cash flows (= S)
    I_expand       : Additional investment required to expand (= K)
    T_years        : Window within which expansion decision must be made (= T)
    r_rf           : Risk-free rate
    sigma_project  : Volatility of the project's cash flows (estimate from comparables)

    Returns
    -------
    dict with: option_value, NPV_static, NPV_with_option
    """
    # ── Treat the option to expand as a call option ────────────────────────────
    call_result = black_scholes(
        S=PV_asset, K=I_expand, T=T_years, r=r_rf, sigma=sigma_project, option="call"
    )

    static_npv       = PV_asset - I_expand          # traditional static NPV
    npv_with_option  = static_npv + call_result["price"]  # expanded NPV

    return {
        "option_value":    call_result["price"],
        "NPV_static":      static_npv,
        "NPV_with_option": npv_with_option,
        "delta":           call_result["delta"],
    }


# ── Case Study: Pharmaceutical Company — Phase 2 Expansion Decision ───────────
print("=" * 60)
print("CASE STUDY: Pharma R&D — Option to Expand to Phase 2")
print("=" * 60)
print()
print("Background:")
print("  A pharma company completes Phase 1 of a drug trial.")
print("  If successful, they can invest £80m to expand to Phase 2.")
print("  The PV of Phase 2 cash flows (if launched) = £75m")
print("  They have 3 years to make the expansion decision.")
print("  Project cash flow volatility (from comparable drugs) ≈ 40%")
print()

result = real_option_expand(
    PV_asset=75,        # PV of Phase 2 cash flows
    I_expand=80,        # cost to expand
    T_years=3,          # decision window
    r_rf=0.04,          # risk-free rate
    sigma_project=0.40  # project volatility
)

print(f"  Static NPV (PV - Investment):       £{result['NPV_static']:.1f}m  ← REJECT by DCF")
print(f"  Option to Expand (BS value):        £{result['option_value']:.1f}m")
print(f"  Expanded NPV (static + option):     £{result['NPV_with_option']:.1f}m  ← ACCEPT with real options")
print()
print(f"  Option Delta: {result['delta']:.2f}  (hedge ratio — how sensitive value is to PV changes)")
print()
print("  Traditional DCF says REJECT. Real options analysis says the FLEXIBILITY")
print("  to expand has positive value — the project should not be abandoned yet.")


In [ ]:

# ── Sensitivity: how option value changes with volatility and time ─────────────
vol_range  = np.linspace(0.10, 0.80, 50)
time_range = [1, 2, 3, 5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Option value vs project volatility ────────────────────────────────────────
for T_yr in time_range:
    vals = [real_option_expand(75, 80, T_yr, 0.04, sig)["option_value"]
            for sig in vol_range]
    axes[0].plot(vol_range * 100, vals, lw=2, label=f"T = {T_yr}yr")

axes[0].set_title("Option to Expand: Value vs Project Volatility", fontweight="bold")
axes[0].set_xlabel("Project Volatility (%)")
axes[0].set_ylabel("Real Option Value (£m)")
axes[0].legend(title="Decision window")

# ── Option value vs PV of asset (different project sizes) ────────────────────
pv_range = np.linspace(30, 140, 100)
for T_yr in [1, 3, 5]:
    vals = [real_option_expand(pv, 80, T_yr, 0.04, 0.40)["option_value"]
            for pv in pv_range]
    axes[1].plot(pv_range, vals, lw=2, label=f"T = {T_yr}yr")

axes[1].axvline(80, color="grey", lw=1, ls="--", label="I = £80m (ATM)")
axes[1].set_title("Option to Expand: Value vs PV of Asset", fontweight="bold")
axes[1].set_xlabel("PV of Future Cash Flows (£m)")
axes[1].set_ylabel("Real Option Value (£m)")
axes[1].legend(title="Decision window")

plt.suptitle("Real Option Sensitivity Analysis  |  Base: PV=75, K=80, σ=40%",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: HIGHER volatility → HIGHER option value.")
print("This is the opposite of DCF thinking — uncertainty creates option value!")


## 8. Bringing It Together — Real Vol from Market Data

Let's use the historical volatility from Session 4's data as the $\sigma$ input to Black-Scholes
and price options on AAPL at various strikes.


In [ ]:

# ── Download AAPL prices and compute realised vol ─────────────────────────────
aapl = (
    yf.download("AAPL", start="2023-01-01", end="2024-12-31",
                auto_adjust=True, progress=False)["Close"]
    .squeeze()
    .rename("AAPL")
    .pipe(lambda s: np.log(s / s.shift(1)))  # log returns
    .dropna()
)

# ── Use last 30-day realised vol as our sigma estimate ────────────────────────
TRADING_DAYS = 252
sigma_30d = aapl.tail(30).std() * np.sqrt(TRADING_DAYS)
S_current = yf.download("AAPL", period="1d", auto_adjust=True, progress=False)["Close"].iloc[-1]
r_current = 0.05   # approximate 10-year UST yield

print(f"AAPL Last Price    : ${S_current:.2f}")
print(f"30-day Realised Vol: {sigma_30d:.1%}")
print(f"Risk-free Rate     : {r_current:.1%}")
print()

# ── Price options at ±20% strikes for 1-month, 3-month, 6-month expiry ────────
expiries = {"1 Month": 1/12, "3 Months": 3/12, "6 Months": 6/12}
K_multipliers = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])

rows_live = []
for label, T_yr in expiries.items():
    for km in K_multipliers:
        K = round(float(S_current) * km, 1)
        bs_c = black_scholes(float(S_current), K, T_yr, r_current, sigma_30d, "call")
        bs_p = black_scholes(float(S_current), K, T_yr, r_current, sigma_30d, "put")
        rows_live.append({
            "Expiry": label, "Strike": K, "K/S": f"{km:.0%}",
            "Call": bs_c["price"], "Put": bs_p["price"],
            "Call Delta": bs_c["delta"], "Call Vega": bs_c["vega"],
        })

live_options = pd.DataFrame(rows_live)

# ── Pivot to option chain style display ───────────────────────────────────────
option_chain = (
    live_options
    .query("Expiry == '3 Months'")
    .assign(Moneyness=lambda df: df["K/S"].map({
        "80%": "Deep ITM", "85%": "ITM", "90%": "ITM",
        "95%": "Slight ITM", "100%": "ATM",
        "105%": "Slight OTM", "110%": "OTM", "115%": "OTM", "120%": "Deep OTM"
    }))
    [["Strike", "K/S", "Call", "Put", "Call Delta", "Call Vega", "Moneyness"]]
    .set_index("Strike")
    .round(3)
)

print("AAPL Option Chain — 3-Month Expiry (BS Theoretical Prices)")
print(option_chain.to_string())


## Summary & Key Takeaways

| Topic | Key Insight |
|-------|-------------|
| Black-Scholes | Closed-form, fast, elegant — but rests on strong assumptions |
| Greeks | Delta = hedge ratio; Gamma = acceleration; Vega = vol sensitivity; Theta = time decay |
| Monte Carlo | Flexible — can price ANY payoff; converges at rate 1/√N |
| Asian options | Path-dependent; no closed form; always cheaper than European |
| Implied vol smile | Markets reject flat-vol BS; smile = fat tails + skew built into prices |
| Real options | Corporate flexibility (expand/abandon/defer) has quantifiable option value |
| Vol estimation | 30-day realised vol is a simple but imperfect estimate of future vol |

---

### Limitations of Black-Scholes

1. **Constant volatility** — volatility actually clusters and smiles (Session 4 showed this)
2. **Continuous trading** — impossible for real assets, illiquid instruments
3. **Log-normal distribution** — ignores fat tails and jump risk (crashes)
4. **No transaction costs** — delta hedging in practice incurs costs
5. **European exercise only** — American options require binomial trees or PDE methods

These limitations motivate more advanced models: **Heston** (stochastic vol),
**SABR** (smile modelling), and **Jump-Diffusion** (Merton model) — natural next steps for further study.


---
---

# 🧪 Real-World Exercises — Pricing Options on Live Stocks

## Overview

In the previous sections we built Black-Scholes and Monte Carlo pricing from first
principles using synthetic parameters.  Now we apply those tools to **real companies**
with very different risk profiles:

| Ticker | Company | Sector | Why interesting |
|--------|---------|--------|----------------|
| **AAPL** | Apple Inc. | Technology | Large-cap, moderate vol, liquid options market |
| **XOM**  | ExxonMobil | Energy | Commodity-linked, macro-sensitive, higher vol |
| **V**    | Visa Inc. | Financials | Stable compounder, low vol, rate-sensitive |
| **DPZ**  | Domino's Pizza | Consumer | Mid-cap, growth stock, higher vol, less intuitive |

These four span a wide range of **implied volatilities**, **betas**, and **option
pricing dynamics** — making them ideal for comparing how the same BS formula
produces very different results depending on the underlying.

---

### What you will build across these exercises

```
Exercise 1 — Download prices & compute realised volatility for all 4 stocks
Exercise 2 — Build a full option chain (calls & puts, multiple strikes & expiries)
Exercise 3 — Visualise the option price surface (strike × expiry heatmap)
Exercise 4 — Analyse the Greeks across the chain; find max-vega and max-gamma strikes
Exercise 5 — Monte Carlo pricing & convergence comparison across all 4 stocks
Exercise 6 — P&L scenarios: what happens to your option if the stock moves ±10%?
Exercise 7 — Implied volatility surface: back out IV from hypothetical market prices
```


## Exercise Setup — Imports & Helper Functions

In [ ]:

# ── All imports needed for the exercises ──────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import product

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)
pd.set_option("display.float_format", "{:.4f}".format)
np.random.seed(42)

# ── Re-use the Black-Scholes and Monte Carlo functions from earlier ────────────
# (These are identical to the implementations above — reproduced here so this
#  exercise section is self-contained and can be run independently)

def black_scholes(S, K, T, r, sigma, option="call"):
    """Black-Scholes price + Greeks for a European option."""
    if T <= 0:
        intrinsic = max(S - K, 0) if option == "call" else max(K - S, 0)
        return {"price": intrinsic, "delta": np.nan, "gamma": np.nan,
                "vega": np.nan, "theta": np.nan, "rho": np.nan}
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
        rho   = K * T * np.exp(-r * T) * norm.cdf(d2)  / 100
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1
        rho   = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100
    if option == "call":
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
    return {"price": price, "d1": d1, "d2": d2,
            "delta": delta, "gamma": gamma,
            "vega": vega, "theta": theta, "rho": rho}


def simulate_gbm(S0, r, sigma, T, n_steps, n_paths):
    """Vectorised GBM path simulation. Returns array (n_steps+1, n_paths)."""
    dt = T / n_steps
    Z  = np.random.standard_normal((n_steps, n_paths))
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths  = np.vstack([np.zeros(n_paths), np.cumsum(increments, axis=0)])
    return S0 * np.exp(log_paths)


def mc_european(S, K, T, r, sigma, n_sims=100_000, option="call"):
    """Monte Carlo price for a European option."""
    Z  = np.random.standard_normal(n_sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0) if option == "call" else np.maximum(K - ST, 0)
    price     = np.exp(-r * T) * payoffs.mean()
    std_error = np.exp(-r * T) * payoffs.std() / np.sqrt(n_sims)
    return {"price": price, "std_error": std_error,
            "ci_lower": price - 1.96 * std_error,
            "ci_upper": price + 1.96 * std_error}


def implied_vol(market_price, S, K, T, r, option="call"):
    """Implied volatility via Brent's method."""
    intrinsic = max(S - K * np.exp(-r * T), 0) if option == "call" else max(K * np.exp(-r * T) - S, 0)
    if market_price <= intrinsic + 1e-8:
        return np.nan
    try:
        return brentq(
            lambda sigma: black_scholes(S, K, T, r, sigma, option)["price"] - market_price,
            1e-6, 10.0, xtol=1e-6
        )
    except ValueError:
        return np.nan


TRADING_DAYS = 252
RISK_FREE_RATE = 0.05    # ~10-year UST yield; adjust if needed
TICKERS = ["AAPL", "XOM", "V", "DPZ"]

print("Setup complete.  Helper functions loaded.")
print(f"Stocks: {TICKERS}")
print(f"Risk-free rate: {RISK_FREE_RATE:.1%}")


---
## Exercise 1 — Download Historical Prices & Compute Realised Volatility

### Your Task
1. Download 2 years of adjusted closing prices for AAPL, XOM, V, and DPZ
2. Compute **daily log-returns** for each stock
3. Compute **realised volatility** using three different lookback windows:
   - 30-day (recent vol — what the market has *just* been doing)
   - 90-day (medium-term)
   - 252-day (full-year — a common input for longer-dated options)
4. Visualise the rolling volatility for all four stocks on one chart
5. Produce a summary table comparing the three vol estimates

> **Why does the lookback window matter?**
> When you plug $\sigma$ into Black-Scholes, you're making a forecast of future volatility.
> The choice of lookback window dramatically changes your option prices — especially for
> stocks like XOM and DPZ where short-term and long-term vol can diverge significantly.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

# ── 1. Download 2 years of prices ─────────────────────────────────────────────
end_date   = pd.Timestamp.today().strftime("%Y-%m-%d")
start_date = (pd.Timestamp.today() - pd.DateOffset(years=2)).strftime("%Y-%m-%d")

raw = yf.download(TICKERS, start=start_date, end=end_date,
                  auto_adjust=True, progress=False)

prices = (
    raw["Close"]
    .rename_axis("Date")
    .rename_axis("Ticker", axis="columns")
    .ffill()
    .dropna(how="all")
)

print(f"Downloaded {prices.shape[0]} trading days × {prices.shape[1]} stocks")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")

# ── 2. Daily log-returns ───────────────────────────────────────────────────────
returns = (
    np.log(prices / prices.shift(1))
    .dropna()
)

# ── 3. Realised vol at three lookback windows ──────────────────────────────────
windows = {"30d": 30, "90d": 90, "252d": 252}

vol_summary = (
    pd.concat(
        {
            label: returns.tail(w).std() * np.sqrt(TRADING_DAYS)
            for label, w in windows.items()
        },
        axis=1
    )
    .rename_axis("Ticker")
    .mul(100)                         # convert to percentage
    .round(1)
    .assign(
        **{"Spread (30d vs 252d)": lambda df: (df["30d"] - df["252d"]).round(1)}
    )
)

print("\nAnnualised Realised Volatility by Lookback Window (%)")
print("=" * 55)
print(vol_summary.to_string())
print()
print("Interpretation:")
print("  Large spread → vol regime recently changed (e.g. an earnings spike)")
print("  Small spread → vol has been stable")


In [ ]:

# ── 4. Rolling volatility chart for all four stocks ───────────────────────────

ROLL_W = 30   # 30-day rolling vol for the chart

roll_vol = (
    returns
    .rolling(ROLL_W)
    .std()
    .mul(np.sqrt(TRADING_DAYS) * 100)
    .dropna()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

palette = {"AAPL": "royalblue", "XOM": "crimson", "V": "seagreen", "DPZ": "darkorange"}

for ax, ticker in zip(axes.flatten(), TICKERS):
    series = roll_vol[ticker]
    ax.plot(series.index, series, lw=1.8, color=palette[ticker])
    ax.fill_between(series.index, series, series.min(), alpha=0.15, color=palette[ticker])

    # Mark the mean and current level
    mean_vol = series.mean()
    curr_vol = series.iloc[-1]
    ax.axhline(mean_vol, color="grey", lw=1, ls="--", alpha=0.7)
    ax.set_title(f"{ticker}  |  Current: {curr_vol:.1f}%  |  Mean: {mean_vol:.1f}%",
                 fontweight="bold")
    ax.set_ylabel("Ann. Vol (%)")

fig.suptitle(f"{ROLL_W}-Day Rolling Annualised Volatility — AAPL, XOM, V, DPZ",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── 5. Store current prices and preferred vol estimates for later exercises ────
current_prices = prices.iloc[-1]
# Use 30-day vol as our "current" sigma — closest to near-term option expiries
current_vols   = returns.tail(30).std() * np.sqrt(TRADING_DAYS)

print("Current prices and 30-day realised vols:")
for t in TICKERS:
    print(f"  {t:<6}  S = ${current_prices[t]:>7.2f}  |  σ(30d) = {current_vols[t]:.1%}")


---
## Exercise 2 — Build a Full Option Chain

### Background
An **option chain** is the grid of call and put prices across multiple **strikes**
and **expiries** that you see on any broker platform (e.g. Interactive Brokers,
Bloomberg, Yahoo Finance options tab).

Strikes are typically quoted as a fraction of the current stock price — so a 95%
strike on a $150 stock is $142.50. This moneyness convention lets you compare
option expensiveness across stocks with very different price levels.

### Your Task
Build a full option chain for **all four stocks** with:
- Strikes at 80%, 85%, 90%, 95%, 100%, 105%, 110%, 115%, 120% of spot
- Expiries at 1 month, 3 months, 6 months, 1 year
- Both calls and puts
- All Greeks computed alongside prices

> **Notice how DPZ options compare to V options at the same moneyness.**
> Higher volatility → more expensive options → higher vega → faster time decay.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

# ── Option chain parameters ────────────────────────────────────────────────────
MONEYNESS   = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
EXPIRIES    = {"1M": 1/12, "3M": 3/12, "6M": 6/12, "1Y": 1.0}
OPTION_TYPE = ["call", "put"]

def build_option_chain(ticker: str, S: float, sigma: float,
                       r: float = RISK_FREE_RATE) -> pd.DataFrame:
    """
    Build a full option chain for one stock.

    Parameters
    ----------
    ticker  : Stock ticker (for labelling)
    S       : Current stock price
    sigma   : Annualised volatility to use (e.g. 30-day realised)
    r       : Risk-free rate

    Returns
    -------
    pd.DataFrame — one row per (expiry × strike × option_type) combination
    """
    rows = []
    for exp_label, T in EXPIRIES.items():
        for km in MONEYNESS:
            K = round(S * km, 2)     # strike = moneyness × spot
            for opt in OPTION_TYPE:
                result = black_scholes(S=S, K=K, T=T, r=r, sigma=sigma, option=opt)
                moneyness_label = (
                    "ATM" if km == 1.00 else
                    ("ITM" if (opt == "call" and km < 1) or (opt == "put" and km > 1) else "OTM")
                )
                rows.append({
                    "Ticker":     ticker,
                    "Expiry":     exp_label,
                    "T (years)":  T,
                    "Strike":     K,
                    "Moneyness":  f"{km:.0%}",
                    "Type":       opt.capitalize(),
                    "ITM/OTM":    moneyness_label,
                    "Price":      round(result["price"], 3),
                    "Delta":      round(result["delta"], 3),
                    "Gamma":      round(result["gamma"], 4),
                    "Vega":       round(result["vega"],  3),
                    "Theta":      round(result["theta"], 3),
                    "Sigma Used": f"{sigma:.1%}",
                })
    return pd.DataFrame(rows)


# ── Build chains for all four stocks ──────────────────────────────────────────
all_chains = pd.concat(
    [build_option_chain(t, current_prices[t], current_vols[t]) for t in TICKERS],
    ignore_index=True
)

print(f"Option chain built: {len(all_chains):,} rows")
print(f"  {len(TICKERS)} stocks × {len(EXPIRIES)} expiries × "
      f"{len(MONEYNESS)} strikes × 2 option types")
print()

# ── Show the 3-month ATM options for all four stocks ──────────────────────────
atm_3m = (
    all_chains
    .query("Expiry == '3M' and Moneyness == '100%'")
    .set_index(["Ticker", "Type"])
    [["Price", "Delta", "Gamma", "Vega", "Theta", "Sigma Used"]]
)

print("ATM 3-Month Options (Call & Put) — All Four Stocks")
print("=" * 65)
print(atm_3m.to_string())
print()
print("Note how Vega and Price scale with volatility:")
print("Higher-vol stocks (DPZ, XOM) have more expensive options.")


In [ ]:

# ── Visualise call prices across strikes for all 4 stocks — 3M expiry ─────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chain_3m_calls = (
    all_chains
    .query("Expiry == '3M' and Type == 'Call'")
    .assign(Moneyness_float=lambda df: df["Moneyness"].str.rstrip("%").astype(float))
)

# ── Left: Absolute option prices ──────────────────────────────────────────────
for ticker, color in palette.items():
    grp = chain_3m_calls.query("Ticker == @ticker")
    axes[0].plot(grp["Moneyness_float"], grp["Price"],
                 "o-", lw=2, color=color, label=ticker)

axes[0].axvline(100, color="grey", lw=1, ls=":", label="ATM")
axes[0].set_title("3-Month Call Prices by Strike", fontweight="bold")
axes[0].set_xlabel("Moneyness (% of Spot)")
axes[0].set_ylabel("Option Price ($)")
axes[0].legend()

# ── Right: Price as % of spot (normalises for different stock prices) ──────────
for ticker, color in palette.items():
    grp = chain_3m_calls.query("Ticker == @ticker").copy()
    grp = grp.assign(price_pct=grp["Price"] / current_prices[ticker] * 100)
    axes[1].plot(grp["Moneyness_float"], grp["price_pct"],
                 "o-", lw=2, color=color, label=ticker)

axes[1].axvline(100, color="grey", lw=1, ls=":", label="ATM")
axes[1].set_title("3-Month Call Prices as % of Spot\n(comparable across stocks)", fontweight="bold")
axes[1].set_xlabel("Moneyness (% of Spot)")
axes[1].set_ylabel("Price / Spot (%)")
axes[1].legend()

plt.suptitle("Option Chain — 3-Month Calls", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## Exercise 3 — The Option Price Surface

### Background
An **option price surface** shows call (or put) prices across **two dimensions simultaneously**:
- Strike (x-axis) — ranging from deep OTM to deep ITM
- Time to expiry (y-axis or heatmap rows) — from 1 month to 1 year

The surface reveals two fundamental option pricing principles:
1. **Time value** — longer-dated options are always worth more (more time for the stock to move)
2. **Intrinsic value** — deep ITM options are mostly intrinsic value; OTM options are purely time value

### Your Task
Create a **heatmap** of call option prices for each stock: rows = expiry, columns = strike.
Then add a second heatmap showing **Vega** (which strike/expiry combination is most
sensitive to a volatility change?).

> **Practical insight:** Traders who expect volatility to rise (but aren't sure about
> direction) will buy the high-vega options — typically ATM, medium-term expiry.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
expiry_order = ["1M", "3M", "6M", "1Y"]

for col_idx, ticker in enumerate(TICKERS):
    for row_idx, (metric, cmap, fmt) in enumerate([
        ("Price", "YlOrRd",   ".2f"),
        ("Vega",  "Blues",    ".3f"),
    ]):
        ax = axes[row_idx, col_idx]

        # ── Pivot to matrix: rows=expiry, columns=moneyness ────────────────────
        pivot = (
            all_chains
            .query("Ticker == @ticker and Type == 'Call'")
            .pivot_table(index="Expiry", columns="Moneyness",
                         values=metric, aggfunc="mean")
            .reindex(expiry_order)              # ensure consistent row order
        )

        sns.heatmap(
            pivot, ax=ax,
            cmap=cmap,
            annot=True, fmt=fmt,
            linewidths=0.5,
            cbar_kws={"shrink": 0.7},
            annot_kws={"size": 8},
        )

        title_metric = f"Call {metric}"
        spot = current_prices[ticker]
        vol  = current_vols[ticker]
        ax.set_title(f"{ticker}  |  S=${spot:.0f}  σ={vol:.0%}\n{title_metric}",
                     fontsize=10, fontweight="bold")
        ax.set_xlabel("Strike (% of Spot)")
        ax.set_ylabel("Expiry" if col_idx == 0 else "")
        ax.tick_params(axis="x", rotation=45)

plt.suptitle("Call Option Price Surface (top) and Vega Surface (bottom)\n"
             "Rows = Expiry, Columns = Moneyness",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  Price surface: prices increase top→bottom (longer expiry) and right→left (deeper ITM)")
print("  Vega surface : highest vega is typically ATM (100%), medium-term expiry")
print("  Compare DPZ vs V: same shape, but DPZ values are uniformly higher due to higher vol")


---
## Exercise 4 — Greeks Analysis: Finding the Most Sensitive Options

### Background
Different traders care about different Greeks:

- **Delta trader** (directional): wants high delta — deep ITM options or short-dated ATM
- **Gamma trader** (volatility of delta): wants high gamma — ATM, short-dated (delta changes fastest)
- **Vega trader** (volatility bet): wants high vega — ATM, long-dated (most exposure to vol changes)
- **Theta seller** (income): sells high-theta options — ATM, short-dated (earns most time decay)

Note: **Long gamma and long theta are opposites** — the option that gains most from large moves
(gamma) also loses most from the passage of time (theta). This is the fundamental options trade-off.

### Your Task
1. For each stock, find the strike and expiry that maximises Gamma (best for vol trading short-term)
2. Find the strike and expiry that maximises Vega (best for a pure vol bet)
3. Plot how Delta evolves across strikes for each stock at 3M expiry
4. Demonstrate the Gamma-Theta trade-off numerically


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

# ── 1 & 2: Find max-Gamma and max-Vega for each stock ─────────────────────────
print("=" * 70)
print(f"{'Ticker':<8} {'Max Gamma Strike':>18} {'Expiry':>8} {'Gamma':>10} {'Theta ($/day)':>15}")
print("=" * 70)

gamma_summary = []
for ticker in TICKERS:
    chain = all_chains.query("Ticker == @ticker and Type == 'Call'")
    idx   = chain["Gamma"].idxmax()
    row   = chain.loc[idx]
    print(f"{ticker:<8} {row['Moneyness']:>18} {row['Expiry']:>8} "
          f"{row['Gamma']:>10.4f} {row['Theta']:>15.4f}")
    gamma_summary.append(row)

print()
print("=" * 70)
print(f"{'Ticker':<8} {'Max Vega Strike':>18} {'Expiry':>8} {'Vega ($/1% vol)':>16} {'Price':>8}")
print("=" * 70)

for ticker in TICKERS:
    chain = all_chains.query("Ticker == @ticker and Type == 'Call'")
    idx   = chain["Vega"].idxmax()
    row   = chain.loc[idx]
    print(f"{ticker:<8} {row['Moneyness']:>18} {row['Expiry']:>8} "
          f"{row['Vega']:>16.3f} {row['Price']:>8.2f}")

print()
print("Observation: max-Vega is always ATM (or near-ATM), longer expiry.")
print("max-Gamma is also ATM but tends toward SHORTER expiry (gamma explodes near expiry).")


In [ ]:

# ── 3. Delta across strikes — 3M expiry, all four stocks ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chain_3m = all_chains.query("Expiry == '3M'").copy()
chain_3m["Moneyness_float"] = chain_3m["Moneyness"].str.rstrip("%").astype(float)

for ticker, color in palette.items():
    grp_call = chain_3m.query("Ticker == @ticker and Type == 'Call'")
    grp_put  = chain_3m.query("Ticker == @ticker and Type == 'Put'")
    axes[0].plot(grp_call["Moneyness_float"], grp_call["Delta"],
                 "o-", lw=2, color=color, label=ticker)
    axes[1].plot(grp_put["Moneyness_float"],  grp_put["Delta"],
                 "o-", lw=2, color=color, label=ticker)

for ax, title in zip(axes, ["Call Delta", "Put Delta"]):
    ax.axvline(100, color="grey", lw=1, ls=":", label="ATM")
    ax.axhline(0,   color="grey", lw=0.7, ls=":")
    ax.set_title(f"3-Month {title} by Strike", fontweight="bold")
    ax.set_xlabel("Moneyness (% of Spot)")
    ax.set_ylabel("Delta")
    ax.legend(fontsize=9)

plt.suptitle("Delta Across Strikes — 3-Month Expiry\n"
             "Higher vol stocks (DPZ, XOM) have a 'flatter' S-curve",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:

# ── 4. Gamma-Theta trade-off: visualise the relationship ─────────────────────

# For ATM options only (moneyness = 100%), across all expiries and stocks
atm_calls = (
    all_chains
    .query("Type == 'Call' and Moneyness == '100%'")
    .assign(Theta_daily=lambda df: df["Theta"])   # already per day
    .assign(Theta_pos=lambda df: -df["Theta"])    # make positive for plotting
)

fig, ax = plt.subplots(figsize=(9, 6))

for ticker, color in palette.items():
    grp = atm_calls.query("Ticker == @ticker").sort_values("T (years)")
    scatter = ax.scatter(
        grp["Gamma"], grp["Theta_pos"],
        c=[color] * len(grp), s=120, zorder=5,
        label=ticker, edgecolors="white", linewidth=0.8
    )
    # Label each point with expiry
    for _, row in grp.iterrows():
        ax.annotate(row["Expiry"],
                    xy=(row["Gamma"], row["Theta_pos"]),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=8, color=color)

ax.set_title("Gamma vs Theta (ATM Calls, All Expiries)\n"
             "High gamma = high time decay — you can't have one without the other",
             fontweight="bold")
ax.set_xlabel("Gamma (rate of change of delta per $1 move)")
ax.set_ylabel("Daily Theta — time decay ($/day, shown as positive loss)")
ax.legend()
plt.tight_layout()
plt.show()

print("The Gamma-Theta trade-off:")
print("  Buying options: you gain from large moves (long gamma)")
print("                  but you lose a little every day (short theta)")
print("  Selling options: you earn daily decay (short theta)")
print("                   but you're exposed to large moves (short gamma)")
print()
print("  Short-dated ATM options: high gamma AND high theta — the most intense trade-off")
print("  Long-dated ATM options:  lower gamma, lower theta  — slower-moving")


---
## Exercise 5 — Monte Carlo Pricing Across All Four Stocks

### Background
Monte Carlo simulation lets us visualise the **distribution of possible outcomes** for
each stock, not just a single option price.  The spread of that distribution is directly
driven by volatility — a higher-vol stock like DPZ has a much wider final price distribution
than a low-vol stock like V.

### Your Task
1. Simulate 50,000 GBM paths for each stock (3-month horizon)
2. Plot the distribution of terminal prices for all four stocks on one chart
3. Price ATM calls and puts using MC and compare to Black-Scholes
4. Show how the terminal price distribution translates into option payoffs

> **Challenge question:** Why does a stock with higher expected return (drift) not
> produce more expensive options under Black-Scholes?
> *(Hint: think about risk-neutral pricing and the role of $r$ vs $\mu$)*


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

N_SIMS = 50_000
T_MC   = 3 / 12   # 3-month horizon
N_STEPS = 63       # ~63 trading days in 3 months

# ── Simulate terminal prices for all four stocks ───────────────────────────────
terminal_prices = {}
for ticker in TICKERS:
    S     = float(current_prices[ticker])
    sigma = float(current_vols[ticker])
    paths = simulate_gbm(S0=S, r=RISK_FREE_RATE, sigma=sigma,
                         T=T_MC, n_steps=N_STEPS, n_paths=N_SIMS)
    terminal_prices[ticker] = paths[-1, :]   # terminal price = last row

# ── Plot terminal price distributions ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, ticker in zip(axes.flatten(), TICKERS):
    S     = float(current_prices[ticker])
    sigma = float(current_vols[ticker])
    ST    = terminal_prices[ticker]
    K_atm = round(S, 0)   # ATM strike = current price

    # Distribution of terminal prices
    sns.histplot(ST, bins=100, kde=True, stat="density",
                 color=palette[ticker], alpha=0.4, ax=ax)

    # Mark key percentiles
    p5, p50, p95 = np.percentile(ST, [5, 50, 95])
    ax.axvline(S,   color="black",          lw=2,   ls="-",  label=f"S₀ = ${S:.0f}")
    ax.axvline(p5,  color="crimson",         lw=1.5, ls="--", label=f"5th pct = ${p5:.0f}")
    ax.axvline(p50, color="grey",            lw=1.5, ls=":",  label=f"Median = ${p50:.0f}")
    ax.axvline(p95, color="seagreen",        lw=1.5, ls="--", label=f"95th pct = ${p95:.0f}")

    # Shade the call payoff region (ST > K)
    x_range   = np.linspace(ST.min(), ST.max(), 300)
    ax.fill_between([K_atm, ST.max()], 0, ax.get_ylim()[1] * 0.001,
                    color="gold", alpha=0.3, label="Call payoff region")

    prob_itm = (ST > K_atm).mean()
    ax.set_title(f"{ticker}  |  σ={sigma:.1%}  |  P(ST>K) = {prob_itm:.1%}",
                 fontweight="bold")
    ax.set_xlabel("Terminal Stock Price ($)")
    ax.legend(fontsize=8)

plt.suptitle(f"Terminal Price Distribution — 3-Month GBM Simulation ({N_SIMS:,} paths)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:

# ── MC vs BS comparison for ATM options — all four stocks ─────────────────────

comparison_rows = []
for ticker in TICKERS:
    S     = float(current_prices[ticker])
    sigma = float(current_vols[ticker])
    K     = round(S, 0)   # ATM strike

    for opt in ["call", "put"]:
        bs_result = black_scholes(S=S, K=K, T=T_MC, r=RISK_FREE_RATE,
                                  sigma=sigma, option=opt)
        mc_result = mc_european(S=S, K=K, T=T_MC, r=RISK_FREE_RATE,
                                sigma=sigma, n_sims=N_SIMS, option=opt)
        comparison_rows.append({
            "Ticker": ticker,
            "Type":   opt.capitalize(),
            "Spot":   f"${S:.2f}",
            "Strike": f"${K:.0f}",
            "σ (30d)": f"{sigma:.1%}",
            "BS Price":  round(bs_result["price"], 3),
            "MC Price":  round(mc_result["price"], 3),
            "MC ±95% CI": f"[{mc_result['ci_lower']:.3f}, {mc_result['ci_upper']:.3f}]",
            "Diff ($)": round(mc_result["price"] - bs_result["price"], 4),
        })

comp_df = pd.DataFrame(comparison_rows).set_index(["Ticker", "Type"])
print("ATM Option Prices — Black-Scholes vs Monte Carlo (50,000 paths)")
print("=" * 85)
print(comp_df.to_string())
print()
print("The BS price should always fall within the MC 95% confidence interval.")
print("Any difference is pure simulation noise — it disappears as N_SIMS → ∞")
print()
print("Answer to challenge question:")
print("  Under risk-neutral pricing, ALL assets earn the risk-free rate r,")
print("  regardless of their real-world expected return μ.")
print("  The stock's actual drift (μ) is irrelevant for option pricing.")
print("  Only σ (volatility) determines the option price, via the spread of the distribution.")


---
## Exercise 6 — P&L Scenario Analysis: What Happens if the Stock Moves?

### Background
Before buying an option, a trader will run **scenario analysis** — "if the stock
moves by X%, what happens to my option value, and what is my P&L?"

This is essentially tracing the option's **payoff profile** at various future points
in time.  The shape of this profile changes dramatically depending on how much time
remains to expiry — this is the effect of **theta** (time decay).

### Your Task
For each stock, price an ATM call today, then compute its value assuming the stock
moves to ±5%, ±10%, ±15%, ±20% of its current level.  Do this for:
- Immediately (t=0): just the intrinsic value change (Delta approximation)
- At 1 month (T-2M remaining for a 3M option)
- At expiry (T=0): the intrinsic payoff max(S-K, 0)

> **This is the "hockey stick" diagram** — a staple of every options primer.
> The interesting part is the curve *before* expiry, where time value makes the
> profile smooth.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

SPOT_MOVES   = np.array([-0.20, -0.15, -0.10, -0.05, 0,
                          0.05,  0.10,  0.15,  0.20])
T_PURCHASE   = 3 / 12    # bought with 3M to expiry
T_SCENARIOS  = {
    "Today (3M left)":  3/12,
    "1M later (2M left)": 2/12,
    "At expiry":        0.0001,   # use tiny T to avoid div-by-zero
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, ticker in zip(axes.flatten(), TICKERS):
    S0    = float(current_prices[ticker])
    sigma = float(current_vols[ticker])
    K     = round(S0, 0)   # ATM strike

    # ── Purchase price (what we paid at T=3M) ─────────────────────────────────
    purchase_price = black_scholes(S=S0, K=K, T=T_PURCHASE, r=RISK_FREE_RATE,
                                   sigma=sigma, option="call")["price"]

    for scenario_label, T_remain in T_SCENARIOS.items():
        pnl_values = []
        spot_values = S0 * (1 + SPOT_MOVES)
        for S_new in spot_values:
            new_price = black_scholes(S=S_new, K=K, T=T_remain, r=RISK_FREE_RATE,
                                      sigma=sigma, option="call")["price"]
            pnl_values.append(new_price - purchase_price)

        ls = "-" if "expiry" in scenario_label else ("--" if "1M" in scenario_label else "-.")
        ax.plot(SPOT_MOVES * 100, pnl_values, lw=2, ls=ls, label=scenario_label)

    # Breakeven line
    ax.axhline(0, color="grey", lw=0.8, ls=":")
    ax.axvline(0, color="grey", lw=0.8, ls=":")

    ax.set_title(f"{ticker}  |  ATM Call (K=${K:.0f})  |  Cost: ${purchase_price:.2f}",
                 fontweight="bold")
    ax.set_xlabel("Stock Move from Purchase Price (%)")
    ax.set_ylabel("P&L per Option ($)")
    ax.legend(fontsize=9)
    ax.set_xlim(-22, 22)

plt.suptitle("Option P&L Scenarios — ATM 3-Month Call\n"
             "Gap between 'Today' and 'At Expiry' lines = Time Value",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  1. At expiry (solid line): classic hockey stick — zero below K, linear above")
print("  2. Before expiry (dashed): smooth curve — time value cushions the downside")
print("  3. The further you are from expiry, the more cushion (higher time value)")
print("  4. DPZ options cost more but also have higher potential P&L for a given move")


---
## Exercise 7 — Building an Implied Volatility Surface

### Background
In real markets, **implied volatility varies by strike and expiry**.  If Black-Scholes
were the complete truth, IV would be flat everywhere.  The fact that it isn't is one
of the most important findings in financial economics:

- **Vol smile**: IV is higher for OTM calls and OTM puts than ATM (symmetric U-shape)
- **Vol skew**: IV is higher for OTM *puts* than OTM *calls* (asymmetric — most equity markets)
- **Term structure**: IV varies across expiries (usually higher for short-dated during stress)

### Your Task
We'll construct a **synthetic implied volatility surface** that mimics real market
behaviour — a volatility skew with term structure — then:
1. Generate "market prices" from this skewed surface
2. Back out the implied vol from each market price
3. Plot the full IV surface as a heatmap
4. Compare the IV surface across the four stocks (higher-vol stocks → higher absolute IV,
   but the *shape* of the skew should be similar)

> **This is the most practically important exercise** — every options desk in the world
> runs some version of this daily to calibrate their pricing models.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────════════════════════════

def parametric_vol_surface(K: float, S: float, T: float,
                            atm_vol: float,
                            skew: float  = -0.10,
                            smile: float =  0.05,
                            term_slope: float = -0.03) -> float:
    """
    Parametric vol surface that generates a realistic skew + term structure.

    Parameters
    ----------
    K          : Strike
    S          : Spot price
    T          : Time to expiry (years)
    atm_vol    : ATM implied vol (annualised)
    skew       : Controls how much IV rises for OTM puts (negative = put skew)
    smile      : Controls U-shape curvature (larger = more smile)
    term_slope : Controls term structure (negative = short-dated vol higher during stress)

    Formula
    -------
    IV(k, T) = atm_vol + skew·k + smile·k² + term_slope·ln(T)
    where k = ln(K/S) is the log-moneyness
    """
    k = np.log(K / S)     # log-moneyness: 0 = ATM, negative = OTM put, positive = OTM call
    t_adj = term_slope * np.log(max(T, 0.01))   # term structure adjustment
    return max(atm_vol + skew * k + smile * k**2 + t_adj, 0.05)  # floor at 5%


# ── Build IV surfaces for all four stocks ─────────────────────────────────────
iv_rows = []
for ticker in TICKERS:
    S       = float(current_prices[ticker])
    atm_vol = float(current_vols[ticker])

    for exp_label, T in EXPIRIES.items():
        for km in MONEYNESS:
            K = round(S * km, 2)

            # ── True IV from parametric surface ────────────────────────────────
            true_iv = parametric_vol_surface(K, S, T, atm_vol)

            # ── "Market price" generated from the true (skewed) IV ─────────────
            market_call_price = black_scholes(S, K, T, RISK_FREE_RATE,
                                              true_iv, "call")["price"]

            # ── Recover IV from market price using BS inversion ─────────────────
            recovered_iv = implied_vol(market_call_price, S, K, T, RISK_FREE_RATE, "call")

            iv_rows.append({
                "Ticker":       ticker,
                "Expiry":       exp_label,
                "T":            T,
                "Strike":       K,
                "Moneyness":    f"{km:.0%}",
                "True IV":      true_iv,
                "Recovered IV": recovered_iv,
                "Flat IV (realised)": atm_vol,
                "IV Skew":      true_iv - atm_vol,   # how far IV deviates from flat
            })

iv_df = pd.DataFrame(iv_rows)

# ── Verify recovery is accurate ────────────────────────────────────────────────
recovery_error = (iv_df["True IV"] - iv_df["Recovered IV"]).abs().max()
print(f"Max IV recovery error: {recovery_error:.2e}  (should be near zero — validates inversion)")


In [ ]:

# ── Plot IV surface heatmaps for all four stocks ───────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
expiry_order = ["1M", "3M", "6M", "1Y"]

for col_idx, ticker in enumerate(TICKERS):
    S       = float(current_prices[ticker])
    atm_vol = float(current_vols[ticker])
    sub_df  = iv_df.query("Ticker == @ticker")

    for row_idx, (col_name, title_suffix, cmap) in enumerate([
        ("True IV",   "Implied Volatility Surface",          "RdYlGn_r"),
        ("IV Skew",   "IV Deviation from Flat (Skew Shape)", "RdBu_r"),
    ]):
        ax = axes[row_idx, col_idx]

        pivot = (
            sub_df
            .pivot_table(index="Expiry", columns="Moneyness",
                         values=col_name, aggfunc="mean")
            .mul(100)            # convert to percentage
            .reindex(expiry_order)
        )

        # For the skew panel, centre the colormap at zero
        center = 0 if "Skew" in col_name else None
        vmax   = pivot.abs().max().max() if "Skew" in col_name else None
        vmin   = -vmax if "Skew" in col_name else None

        sns.heatmap(
            pivot, ax=ax,
            cmap=cmap, center=center,
            vmin=vmin, vmax=vmax,
            annot=True, fmt=".1f",
            linewidths=0.5,
            cbar_kws={"shrink": 0.7, "label": "%"},
            annot_kws={"size": 8},
        )

        ax.set_title(f"{ticker}  (ATM vol={atm_vol:.0%})\n{title_suffix}",
                     fontsize=9, fontweight="bold")
        ax.set_xlabel("Strike (% of Spot)")
        ax.set_ylabel("Expiry" if col_idx == 0 else "")
        ax.tick_params(axis="x", rotation=45)

plt.suptitle("Implied Volatility Surfaces (top) and Skew Deviations (bottom)\n"
             "IV (%) by Expiry × Strike",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Interpreting the IV surface:")
print("  Green cells (top panel): lower IV — options are relatively cheap")
print("  Red cells   (top panel): higher IV — options are relatively expensive")
print()
print("  Blue cells (bottom panel): IV < flat (OTM calls are cheap relative to ATM)")
print("  Red cells  (bottom panel): IV > flat (OTM puts are expensive = 'put skew')")
print()
print("  All four stocks show the same qualitative skew shape, but the absolute")
print("  IV levels differ dramatically — reflecting each stock's risk profile.")


In [ ]:

# ── Visualise the vol smile / skew as a line chart ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: IV smile at each expiry (one stock — DPZ has most pronounced skew) ───
for ticker, color in palette.items():
    sub_3m = iv_df.query("Ticker == @ticker and Expiry == '3M'")
    km_float = sub_3m["Moneyness"].str.rstrip("%").astype(float)
    axes[0].plot(km_float, sub_3m["True IV"] * 100,
                 "o-", lw=2, color=color, label=ticker)

axes[0].axvline(100, color="grey", lw=1, ls=":", label="ATM")
axes[0].set_title("Implied Volatility Smile — 3-Month Expiry", fontweight="bold")
axes[0].set_xlabel("Moneyness (% of Spot)")
axes[0].set_ylabel("Implied Volatility (%)")
axes[0].legend()

# ── Right: IV term structure for ATM options ──────────────────────────────────
T_map = {"1M": 1/12, "3M": 3/12, "6M": 6/12, "1Y": 1.0}
for ticker, color in palette.items():
    sub_atm = iv_df.query("Ticker == @ticker and Moneyness == '100%'")
    t_vals  = sub_atm["T"].values
    iv_vals = sub_atm["True IV"].values * 100
    sort_idx = np.argsort(t_vals)
    axes[1].plot(t_vals[sort_idx], iv_vals[sort_idx],
                 "o-", lw=2, color=color, label=ticker)

axes[1].set_title("ATM IV Term Structure", fontweight="bold")
axes[1].set_xlabel("Time to Expiry (years)")
axes[1].set_ylabel("ATM Implied Volatility (%)")
axes[1].set_xticks([1/12, 3/12, 6/12, 1.0])
axes[1].set_xticklabels(["1M", "3M", "6M", "1Y"])
axes[1].legend()

plt.suptitle("Volatility Smile and Term Structure\n"
             "IV is not flat — it varies by strike and expiry",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Real-world context:")
print("  The put skew (OTM puts more expensive than OTM calls) reflects")
print("  demand for crash insurance — investors pay a premium to hedge downside.")
print()
print("  Downward-sloping term structure means short-dated vol is higher —")
print("  this typically happens after a spike (e.g. earnings, macro shock)")
print("  when markets expect volatility to mean-revert.")


---
## 🏁 Exercise Summary

| Exercise | What you built | Core concept |
|----------|---------------|-------------|
| **1** | Realised vol at 3 lookback windows | Vol is not constant — lookback choice matters |
| **2** | Full option chain (4 stocks × 4 expiries × 9 strikes × call/put) | How strike and expiry affect price |
| **3** | Price surface and Vega surface heatmaps | Time value increases with expiry; vega peaks ATM |
| **4** | Greeks analysis; max-gamma and max-vega strikes; Gamma-Theta trade-off | No free lunch — gamma and theta are inseparable |
| **5** | MC terminal distributions; BS vs MC comparison | Risk-neutral pricing; vol drives prices, drift does not |
| **6** | P&L scenario profiles at different points in time | Hockey stick payoff; time value as a cushion |
| **7** | IV surface construction and recovery; smile and term structure | Real markets don't have flat vol — skew is pervasive |

---

### What to read next
- **Hull, *Options, Futures and Other Derivatives*** — the standard textbook; chapters 13–19 cover all of this material rigorously
- **Natenberg, *Option Volatility and Pricing*** — the practitioner's bible for intuition on Greeks and vol
- **Gatheral, *The Volatility Surface*** — for a deeper treatment of smile modelling (SV, SABR, etc.)

### Extensions for further study
- **American options** — require binomial trees or finite-difference PDE methods (early exercise changes everything)
- **Stochastic volatility** — Heston model: $d\sigma^2 = \kappa(\theta - \sigma^2)dt + \xi \sigma dW^\sigma$
- **Jump-diffusion** — Merton model: adds Poisson jump process to GBM to capture crash risk
- **Least-Squares Monte Carlo (LSM)** — Longstaff-Schwartz algorithm for pricing American options by simulation
